# Чат-бот на базе rugpt3small_based_on_gpt2

## Среда

### Установка бибилиотек

In [1]:
# pip install transformers datasets torch rouge detoxify nltk bert-score

### Стандартные бибилотеки

In [2]:
import time
import re
import json

### Обработка данных

In [3]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

### Нейросеть

In [4]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

### Визуализация

In [5]:
from tqdm.notebook import tqdm

### Метрики

In [6]:
from rouge import Rouge
from rouge_score import rouge_scorer
from detoxify import Detoxify
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score

### Чекпоинт

In [7]:
# checkpoint_path = "./results/checkpoint-2500/"

## Данные

### Загрузка датасета

In [8]:
dataset = load_dataset("MLNavigator/russian-retrieval")

### Очистка датасета

In [9]:
# Очистка датасета от меток SOURCE
def clean_dataset(example):
    example['q'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['q'], flags=re.IGNORECASE).strip()
    example['a'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['a'], flags=re.IGNORECASE).strip()
    return example

dataset = dataset.map(clean_dataset)

### Уменьшение датасета для ускорения тестирования

In [10]:
dataset['train'] = dataset['train'].select(range(1000))

### Предобработка

In [11]:
# Разделение на train/val/test (80%/10%/10%)
split_dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_val_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

train_data = split_dataset['train']
val_data = train_val_split['train']
test_data = train_val_split['test']

# Подготовка данных
tokenizer = GPT2Tokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Установка pad_token

def preprocess_function(examples):
    inputs = [f"Вопрос: {q} Ответ: {a}" for q, a in zip(examples['q'], examples['a'])]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_attention_mask=True  # Добавить маску внимания
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Предобработка на данных
train_data = train_data.map(preprocess_function, batched=True)
val_data = val_data.map(preprocess_function, batched=True)
test_data = test_data.map(preprocess_function, batched=True)

In [12]:
print(len(train_data), len(val_data), len(test_data))

800 100 100


## Модель

### Загрузка пердобученной модели

In [13]:
model = GPT2LMHeadModel.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

In [14]:
# # Восстановление весов lm_head
# if hasattr(model, "transformer") and hasattr(model.transformer, "wte"):
#     model.lm_head.weight.data = model.transformer.wte.weight.data.clone()

# # Проверка наличия весов
# print("lm_head.weight" in model.state_dict())

### Оптимизация для GPU или CPU

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
device

'cuda'

In [16]:
# Очистка кэша PyTorch
torch.cuda.empty_cache()

### Настройка параметров обучения с учётом параметров RTX2060

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    fp16=True if device == "cuda" else False,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to= "none",
)

### Обучение модели

In [18]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-2 не использует masked language modeling
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,  # data_collator вместо tokenizer
)

trainer.train()
trainer.save_model(".results/final_model")


Step,Training Loss,Validation Loss
200,1.399700,2.183673


In [19]:
# Сохранения логов обучения
training_logs = trainer.state.log_history

# Сохранение в DataFrame и CSV
df_logs = pd.DataFrame([log for log in training_logs if 'loss' in log or 'eval_loss' in log])
df_logs.to_csv("rugpt_training_logs.csv", index=False)

print("Результаты обучения сохранены в rugpt_training_logs.csv")
df_logs

Результаты обучения сохранены в rugpt_training_logs.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,10.5760,270.171295,0.000003,0.2,10,NaN,NaN,NaN,NaN
1,8.0517,96.407089,0.000008,0.4,20,NaN,NaN,NaN,NaN
2,5.5647,84.696785,0.000013,0.6,30,NaN,NaN,NaN,NaN
3,4.1549,48.740353,0.000018,0.8,40,NaN,NaN,NaN,NaN
4,3.2493,33.082809,0.000023,1.0,50,NaN,NaN,NaN,NaN
5,2.7575,23.804747,0.000028,1.2,60,NaN,NaN,NaN,NaN
6,2.5277,25.601307,0.000033,1.4,70,NaN,NaN,NaN,NaN
7,2.5057,19.067762,0.000038,1.6,80,NaN,NaN,NaN,NaN
8,2.3576,34.862137,0.000043,1.8,90,NaN,NaN,NaN,NaN
9,2.3516,17.936621,0.000048,2.0,100,NaN,NaN,NaN,NaN


### Генерация ответа

In [20]:
# Функиця генерации ответа
def generate_answer(question):
    try:
        model.eval()
        input_text = f"Вопрос: {question} Ответ:"
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            return_attention_mask=True  # Генерация маски внимания
        ).to(device)

        start = time.time() # время начала ответа
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],  # Передача маски внимания
                max_new_tokens=50,
                num_return_sequences=1,
                no_repeat_ngram_size=2,
                num_beams=5,
                # do_sample=True,
                # # top_k=10,
                # temperature=0.7,
                # top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        clean_response = re.sub(r'\n+.*', '', response, flags=re.IGNORECASE)
        generated_answer = clean_response.split('Ответ:')[-1].strip()

        latency = time.time() - start # длительность ответа
        
        return generated_answer, latency
            
    except Exception as e:
        print(f"Ошибка генерации: {e}")
        return "Не удалось сгенерировать ответ", 0.0 

# Функция генерации и сохранения ответов на тестовом датасете
def generate_answers_and_save(test_data, output_file="rugpt_generated_answers.json"):
    generated_data = []
    
    for example in tqdm(test_data, desc="Генерация ответов", unit="example", total=len(test_data)):
        q = example['q']
        true_answer = example['a']
        
        # Генерация ответа
        answer, latency = generate_answer(q)
        
        # Запись для сохранения
        generated_data.append({
            "Вопрос": q,
            "Эталонный ответ": true_answer,
            "Сгенерированный ответ": answer,
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Сохранение в файл
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(generated_data, f, ensure_ascii=False, indent=2)
    print(f"Сгенерированные ответы сохранены в {output_file}")

In [21]:
# Запуск генерации ответов
generate_answers_and_save(test_data)

Генерация ответов:   0%|          | 0/100 [00:00<?, ?example/s]

Сгенерированные ответы сохранены в rugpt_generated_answers.json


## Оценка качества модели

### Загрузка моделей

In [22]:
# Загрузка модели Detoxify для оценки токсичности ответов
detox = Detoxify('original', device=device)
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Функции для расчета метрик

In [23]:
# Перплексия для оценки предсказания текста
def calculate_perplexity(question, generated_answer):
    inputs = tokenizer(f"Вопрос: {question} Ответ: {generated_answer}", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

In [24]:
def calculate_bertscore(generated, true_answer):
    P, R, F1 = score(
        [generated], 
        [true_answer], 
        lang="ru", 
        model_type="bert-base-multilingual-cased",
        verbose=False
    )
    return F1.item()

In [25]:
# Токсичность ответов
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Расчет метрик из сохраненных данных

In [26]:
# Функция расчёта метрик 
def calculate_metrics_from_file(input_file="rugpt_generated_answers.json", output_file="rugpt_metrics.csv"):
    with open(input_file, "r", encoding="utf-8") as f:
        generated_data = json.load(f)
    
    metrics = []
    
    for example in tqdm(generated_data, desc="Расчет метрик", unit="example", total=len(generated_data)):
        q = example["Вопрос"]
        true_answer = example["Эталонный ответ"]
        generated = example["Сгенерированный ответ"]
        latency = example.get("Время отклика (сек)", 0.0)
        
        # Проверка, что на входе строка
        if not isinstance(generated, str):
            generated = " ".join(generated)
        
        # Расчет метрик
        ppl = calculate_perplexity(q, generated) 
        # sbert_similarity = calculate_sbert_similarity(generated, true_answer)
        bert_score = calculate_bertscore(generated, true_answer)
        # rouge1, rouge2, rougel = calculate_rouge(generated, true_answer)
        toxicity = check_toxicity(generated)
        
        metrics.append({
            "Вопрос": q,
            "Сгенерированный ответ": generated,
            "Эталонный ответ": true_answer,
            "Perplexity": round(ppl, 2),
            "BERTScore F1": round(bert_score, 4),
            "Токсичность": round(toxicity, 4),
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Создание DataFrame и сохранение
    df = pd.DataFrame(metrics)
    df.to_csv(output_file, index=False)
    print(f"Метрики сохранены в {output_file}")

    avg_latency = df["Время отклика (сек)"].mean()

    # Сводка по средним значениям
    summary = df.mean(numeric_only=True)
    print("Средние значения метрик:")
    print(summary)

### Расчёт метрик

In [27]:
calculate_metrics_from_file()

Расчет метрик:   0%|          | 0/100 [00:00<?, ?example/s]

Метрики сохранены в rugpt_metrics.csv
Средние значения метрик:
Perplexity             4.440300
BERTScore F1           0.677267
Токсичность            0.002387
Время отклика (сек)    1.189658
dtype: float64


## Чат-бот

In [28]:
def chat_bot():
    print("Бот: Здравствуйте! Для выхода введите 'выход'.\n")
    qa_history_correct = []  # История правильных ответов
    qa_history_incorrect = []  # История неправильных ответов
    
    while True:
        question = input("Вы: ")
        if question.lower() == "выход":
            # Сохранение в соотвествующие файлы
            with open("rugpt_qa_correct.json", "w", encoding="utf-8") as f:
                json.dump(qa_history_correct, f, ensure_ascii=False, indent=2)
            with open("rugpt_qa_incorrect.json", "w", encoding="utf-8") as f:
                json.dump(qa_history_incorrect, f, ensure_ascii=False, indent=2)
            print("Бот: До свидания!")
            break
        
        # Генерация ответа через функцию выше
        answer, latency = generate_answer(question)
        print(f"Бот: {answer}, {round(latency, 2)} сек")
        
        # Обратная связь
        while True:
            feedback = input("ℹ️ Вас устраивает ответ? (да/нет): ").lower()
            if feedback in ["да", "нет"]:
                break
            print("Введите 'да' или 'нет'")
        
        if feedback == "да":
            qa_history_correct.append({
                "question": question,
                "answer": answer,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
            })
            print("✅ Правильный ответ сохранен в rugpt_qa_correct.json\n")
        else:
            qa_history_incorrect.append({
                "question": question,
                "incorrect_answer": answer,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
            })
            print("❌ Неправильный ответ сохранен в rugpt_qa_incorrect.json\n")

# Запуск чат-бота
if __name__ == "__main__":
    chat_bot()
    

Бот: Здравствуйте! Для выхода введите 'выход'.



Вы:  Кто президент РФ?


Бот: первый заместитель председателя правительства РФ Дмитрий Медведев, отвечая на вопрос о том, что происходит в экономике и социальной сфере, сказал: &laquo;Уровень жизни населения в России в 2010 году достиг исторического максимума за последние 20 лет&hellip, 1.23 сек


ℹ️ Вас устраивает ответ? (да/нет):  нет


❌ Неправильный ответ сохранен в rugpt_qa_incorrect.json



...

Бот: белки живут в тропиках и субтропиках в умеренных водах Мирового океана. В Гренландии и некоторых районах Крайнего Севера живут белки-альбиносы и другие арктические экваториальные млекопитающие., 1.06 сек


ℹ️ Вас устраивает ответ? (да/нет):  нет


❌ Неправильный ответ сохранен в rugpt_qa_incorrect.json



Вы:  выход


Бот: До свидания!
